# Lab 03 — Inciso d) Comparación entre familias de algoritmos de regresión
**EIF420O Inteligencia Artificial — UNA — I Ciclo 2026**

Dataset: Diabetes (sklearn) — 442 observaciones, 10 predictores, variable objetivo continua.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso, LassoCV
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

%matplotlib inline
plt.rcParams.update({"font.size": 10, "figure.dpi": 120})

## 1. Carga y exploración del dataset

In [ ]:
df = pd.read_csv("diabetes_regression_lab03.csv")
TARGET = "target"
print(f"Shape: {df.shape}")
print(f"Target: min={df[TARGET].min():.1f}, max={df[TARGET].max():.1f}, mean={df[TARGET].mean():.2f}, std={df[TARGET].std():.2f}")
print(f"Missing values: {df.isna().sum().sum()}")
df.describe().round(4)

## 2. Preparación de datos

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print(f"Train: {X_train_sc.shape[0]} | Test: {X_test_sc.shape[0]}")

## 3. Definición de modelos por familia

In [ ]:
MODELS = [
    # Family: Lineal
    ("LinearRegression",        "Lineal",       LinearRegression()),
    ("Ridge (α=1)",             "Lineal",       Ridge(alpha=1.0)),
    ("RidgeCV",                 "Lineal",       RidgeCV(alphas=np.logspace(-4, 4, 30))),
    ("Lasso (α=0.01)",         "Lineal",       Lasso(alpha=0.01, max_iter=20000)),
    ("LassoCV",                 "Lineal",       LassoCV(max_iter=20000, cv=5)),
    # Family: Kernel/SVM
    ("SVR (rbf)",               "Kernel/SVM",   SVR(kernel="rbf", C=100, epsilon=0.1, gamma="scale")),
    ("SVR (linear)",            "Kernel/SVM",   SVR(kernel="linear", C=100, epsilon=0.1)),
    ("SVR (poly)",              "Kernel/SVM",   SVR(kernel="poly", C=100, degree=3, epsilon=0.1)),
    # Family: Árbol
    ("DecisionTree (std)",      "Árbol",        DecisionTreeRegressor(random_state=42)),
    ("DecisionTree (podado)",   "Árbol",        DecisionTreeRegressor(max_depth=4, min_samples_split=10, min_samples_leaf=5, random_state=42)),
    # Family: Ensamble
    ("RandomForest (100)",      "Ensamble",     RandomForestRegressor(n_estimators=100, random_state=42)),
    ("RandomForest (reg)",      "Ensamble",     RandomForestRegressor(n_estimators=120, max_depth=5, min_samples_leaf=4, max_features="sqrt", random_state=42)),
    ("GradientBoosting",        "Ensamble",     GradientBoostingRegressor(n_estimators=120, learning_rate=0.05, max_depth=2, subsample=0.9, random_state=42)),
]

if HAS_XGB:
    MODELS.append(("XGBoost", "Ensamble", XGBRegressor(n_estimators=120, learning_rate=0.05, max_depth=2, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0, random_state=42, n_jobs=1)))

print(f"Total models: {len(MODELS)}")

## 4. Entrenamiento, validación cruzada y evaluación

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"neg_rmse": "neg_root_mean_squared_error", "neg_mae": "neg_mean_absolute_error", "r2": "r2"}

rows = []
fitted_models = {}

for name, family, model in MODELS:
    cv_res = cross_validate(model, X_train_sc, y_train, cv=cv, scoring=scoring, return_train_score=False)
    cv_rmse = -cv_res["test_neg_rmse"].mean()
    cv_mae = -cv_res["test_neg_mae"].mean()
    cv_r2 = cv_res["test_r2"].mean()

    fitted = clone(model).fit(X_train_sc, y_train)
    y_pred = fitted.predict(X_test_sc)

    test_mse = float(mean_squared_error(y_test, y_pred))
    test_rmse = float(np.sqrt(test_mse))
    test_mae = float(mean_absolute_error(y_test, y_pred))
    test_r2 = float(r2_score(y_test, y_pred))

    rows.append({"Modelo": name, "Familia": family, "CV_RMSE": round(cv_rmse, 4), "CV_MAE": round(cv_mae, 4),
                 "CV_R2": round(cv_r2, 4), "Test_MSE": round(test_mse, 2), "Test_RMSE": round(test_rmse, 4),
                 "Test_MAE": round(test_mae, 4), "Test_R2": round(test_r2, 4)})
    fitted_models[name] = {"model": fitted, "y_pred": y_pred, "family": family}

benchmark = pd.DataFrame(rows).sort_values("CV_RMSE").reset_index(drop=True)
benchmark

## 5. Mejor modelo por familia

In [ ]:
best_per_family = benchmark.sort_values("CV_RMSE").groupby("Familia", as_index=False).first().sort_values("CV_RMSE")
best_per_family

## 6. Visualizaciones

In [ ]:
colors = {"Lineal": "#2196F3", "Kernel/SVM": "#FF9800", "Árbol": "#4CAF50", "Ensamble": "#9C27B0"}
from matplotlib.patches import Patch
handles = [Patch(facecolor=c, label=f) for f, c in colors.items()]

# RMSE bar chart
fig, ax = plt.subplots(figsize=(10, 6))
data = benchmark.sort_values("Test_RMSE", ascending=True)
ax.barh(data["Modelo"], data["Test_RMSE"], color=[colors.get(f, "#999") for f in data["Familia"]])
ax.set_xlabel("Test RMSE"); ax.set_title("Comparación entre familias: RMSE en prueba")
ax.legend(handles=handles, loc="lower right"); ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# R² bar chart
fig, ax = plt.subplots(figsize=(10, 6))
data2 = benchmark.sort_values("Test_R2", ascending=True)
ax.barh(data2["Modelo"], data2["Test_R2"], color=[colors.get(f, "#999") for f in data2["Familia"]])
ax.set_xlabel("Test R²"); ax.set_title("Comparación entre familias: R² en prueba")
ax.legend(handles=handles, loc="lower right"); ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Best per family
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bpf = best_per_family.sort_values("Test_RMSE", ascending=True)
axes[0].barh(bpf["Familia"] + "\n(" + bpf["Modelo"] + ")", bpf["Test_RMSE"], color=[colors.get(f, "#999") for f in bpf["Familia"]])
axes[0].set_xlabel("Test RMSE"); axes[0].set_title("Mejor modelo por familia — RMSE"); axes[0].grid(axis="x", alpha=0.3)

bpf2 = best_per_family.sort_values("Test_R2", ascending=True)
axes[1].barh(bpf2["Familia"] + "\n(" + bpf2["Modelo"] + ")", bpf2["Test_R2"], color=[colors.get(f, "#999") for f in bpf2["Familia"]])
axes[1].set_xlabel("Test R²"); axes[1].set_title("Mejor modelo por familia — R²"); axes[1].grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# CV vs Test RMSE scatter
fig, ax = plt.subplots(figsize=(8, 6))
for fam, grp in benchmark.groupby("Familia"):
    ax.scatter(grp["CV_RMSE"], grp["Test_RMSE"], s=80, label=fam, color=colors.get(fam, "#999"), edgecolors="k", linewidth=0.5)
    for _, row in grp.iterrows():
        ax.annotate(row["Modelo"], (row["CV_RMSE"], row["Test_RMSE"]), fontsize=6.5, alpha=0.8, xytext=(4, 4), textcoords="offset points")
mn = min(benchmark["CV_RMSE"].min(), benchmark["Test_RMSE"].min()) - 2
mx = max(benchmark["CV_RMSE"].max(), benchmark["Test_RMSE"].max()) + 2
ax.plot([mn, mx], [mn, mx], "--", color="gray", alpha=0.5)
ax.set_xlabel("CV RMSE"); ax.set_ylabel("Test RMSE")
ax.set_title("CV RMSE vs Test RMSE — Estabilidad entre familias"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Análisis del mejor modelo global

In [ ]:
overall_best = benchmark.iloc[0]["Modelo"]
print(f"Mejor modelo global: {overall_best} (Familia: {benchmark.iloc[0]['Familia']})")
print(benchmark.iloc[0].to_frame().T.to_string(index=False))

# Predicted vs Real
y_pred_best = fitted_models[overall_best]["y_pred"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred_best, alpha=0.6, edgecolors="k", linewidth=0.3)
mn, mx = min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())
axes[0].plot([mn, mx], [mn, mx], "--", color="red"); axes[0].set_xlabel("Valor real"); axes[0].set_ylabel("Valor predicho")
axes[0].set_title(f"Predicho vs. Real — {overall_best}"); axes[0].grid(alpha=0.3)

residuals = y_test.values - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.6, edgecolors="k", linewidth=0.3)
axes[1].axhline(0, linestyle="--", color="red")
axes[1].set_xlabel("Valor predicho"); axes[1].set_ylabel("Residuo")
axes[1].set_title(f"Residuos — {overall_best}"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Heatmap
metrics_fam = best_per_family.set_index("Familia")[["Test_RMSE", "Test_MAE", "Test_R2"]].copy()
metrics_fam.columns = ["RMSE", "MAE", "R²"]
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(metrics_fam, annot=True, fmt=".2f", cmap="YlOrRd_r", ax=ax, linewidths=0.5)
ax.set_title("Métricas del mejor modelo por familia")
plt.tight_layout(); plt.show()

## 8. Conclusión

La comparación entre familias demuestra que los **modelos lineales** (Ridge, Lasso, LinearRegression) ofrecen el mejor equilibrio entre estabilidad en validación cruzada y desempeño en prueba para el dataset Diabetes. **Ridge (α=1)** fue seleccionado como mejor modelo global con un CV RMSE de 56.07 y Test R² de 0.4859.

Los ensambles (RandomForest, GradientBoosting, XGBoost) capturan señales no lineales parciales pero no superan la consistencia de los lineales en un dataset de 442 observaciones con estructura fundamentalmente lineal. SVR con kernel RBF obtuvo el menor error puntual en prueba, pero con menor estabilidad en CV. Los árboles individuales fueron la familia menos competitiva.